In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, DoubleType

def log_pipeline_run(pipeline_name, task_name, started_at, completed_at,
                      records_read, records_written, records_rejected,
                      status, error_message=None):
    run_id = str(uuid.uuid4())
    duration_seconds = (completed_at - started_at).total_seconds()

    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("task_name", StringType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("completed_at", TimestampType(), True),
        StructField("records_read", LongType(), True),
        StructField("records_written", LongType(), True),
        StructField("records_rejected", LongType(), True),
        StructField("duration_seconds", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
    ])

    row = [(
        run_id, pipeline_name, task_name, started_at, completed_at,
        records_read, records_written, records_rejected,
        duration_seconds, status, error_message
    )]

    df = spark.createDataFrame(row, schema=schema)
    df.write.format("delta").mode("append").saveAsTable("urban_mobility.monitoring.pipeline_runs")
    print(f"Logged run {run_id}: {task_name} -> {status} ({duration_seconds:.1f}s)")
    return run_id